In [ ]:
from Exoplanet import Exoplanet
import datetime
import pandas
import os

data_paths = []

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

for path in data_paths:
    e = Exoplanet.load_from_json(path)

    url = f'https://exofop.ipac.caltech.edu/tess/download_stellar.php?id={e.id_num}'
    data = pandas.read_csv(url, delimiter = '|')

    e.add_host_star_attributes(
        radius = data['Radius (R_Sun)'].to_list()[0],
        mass = data['Mass (M_Sun)'].to_list()[0],
        teff = data['Teff (K)'].to_list()[0]
    )

    print(data)

    e.calculate_attributes()

    e.to_json(f'Exoplanet Discovering/Paper Materials/Data/{e.host_star['name']} Exoplanet {e.letter}.json')

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    file.truncate(0)

  Table  ID  Telescope  Instrument  Spectral Type  Teff (K)  Teff (K) Error  \
0   TIC NaN        NaN         NaN            NaN      4842             124   

   log(g)  log(g) Error  Radius (R_Sun)  ...  Distance (pc) Error  Age (Gyr)  \
0     NaN           NaN         18.5659  ...               52.575        NaN   

   Age (Gyr) Error  SNR  Fit Quality        Date               User  Group  \
0              NaN  NaN          NaN  2019-04-15  Exoplanet Archive    NaN   

   Tag     Notes  
0  NaN  TIC v8.2  

[1 rows x 43 columns]


In [55]:
import os
import json

data = []

for (dirpath, dirnames, filenames) in os.walk('Exoplanet Discovering/Paper Materials/Data'):
    for filepath in filenames:
        if filepath == '.DS_Store' or filepath.endswith('.csv'):
            continue

        with open(dirpath + '/' + filepath, 'r') as file:
            exoplanet = json.loads(file.read())

        if type(exoplanet) == list:
            continue

        data.append(exoplanet)

    break

with open('Exoplanet Discovering/Paper Materials/params_planet_20250430_001.txt', 'w') as file:
    for exoplanet in data:
        file.write(exoplanet['csv_string'] + '\n')

In [56]:
import os
import json

data = []

for (dirpath, dirnames, filenames) in os.walk('Exoplanet Discovering/Paper Materials/Data'):
    for filepath in filenames:
        if filepath == '.DS_Store' or filepath.endswith('.csv'):
            continue

        with open(dirpath + '/' + filepath, 'r') as file:
            exoplanet = json.loads(file.read())

        if type(exoplanet) == list:
            continue

        data.append(exoplanet)

    break

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'w') as file:
    string = json.dumps(data, indent = 4)
    string = string.replace('NaN', 'null')
    file.write(string)

In [ ]:
import json
import pandas
import math

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['parameters'] for exoplanet in data])
data['target'] = data['target'].apply(lambda x: x.replace('TIC', 'TIC '))
data = data.set_index('target')
data.replace('', math.nan, inplace = True)
data = data.drop(['tag', 'r_planet', 'flag'], axis = 1)
data['disp'] = data['disp'].where(data['radius'] < 22.42, 'FP')
data = data.sort_values(['disp', 'radius'], ascending = [True, False])

data = data.drop(['TIC ' + str(ticid) + '.01' for ticid in [
        284611896,
        268290940,
        62573638,
        23689332,
        441444256,
        234518605,
        306635088,
        451596415]])

columns = ['disp', 'period', 'period_unc', 'radius', 'mass', 'temp', 'depth', 'depth_unc', 'duration', 'duration_unc', 'epoch', 'epoch_unc', 'sma']
data = data.loc[:, columns]
data = data.round(4)
data.replace(math.nan, 'null', inplace = True)
data.to_csv('Exoplanet Discovering/Paper Materials/Data/ctois.csv')

['TIC 451596415: 1.93195', 'TIC 441444256: None', 'TIC 102625324: 1.07497', 'TIC 467113954: 1.23924', 'TIC 268290940: 1.55678', 'TIC 90083037: 2.08196', 'TIC 73248090: 1.7265', 'TIC 35655828: 3.26579', 'TIC 234518605: 1.04584', 'TIC 62573638: 2.50343', 'TIC 300291839: 2.21022', 'TIC 375477302: 1.46202', 'TIC 76903841: 1.96963', 'TIC 295440174: 2.63408', 'TIC 176871438: 2.56582', 'TIC 35655828: 3.26579', 'TIC 55655482: 1.18042', 'TIC 279322914: 1.24616', 'TIC 342556321: 2.17367', 'TIC 284611896: None', 'TIC 270462117: 1.79504', 'TIC 55727842: 0.298092', 'TIC 734505586: 1.07141', 'TIC 306635088: 2.14238', 'TIC 11799872: 1.69338', 'TIC 38850860: 1.15758', 'TIC 297701617: 2.13541', 'TIC 23689332: 1.37238']


/var/folders/jc/h35mhndj30l_xk52p_1lqvxh0000gn/T/ipykernel_3757/720242378.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace('', math.nan, inplace = True)
